# Malicious URL Detector — Training Pipeline (trimmed)

Clean, linear end-to-end: load → harmonize labels → extract features → train RF → save model.

The MLP and PhiUSIIL-raw-feature experiments from earlier iterations have been removed — they weren't used by the final model and slowed down re-runs. If you need them, refer to the original notebook in version control.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from urllib.parse import urlparse

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


## 1. Load PhiUSIIL

In [ ]:
import kagglehub

path = kagglehub.dataset_download("ndarvind/phiusiil-phishing-url-dataset")
df_phi = pd.read_csv(f"{path}/PhiUSIIL_Phishing_URL_Dataset.csv")

print("PhiUSIIL shape:", df_phi.shape)
print(df_phi["label"].value_counts())


## 2. Load real-world dataset

In [ ]:
df_rw = pd.read_csv("url_data_mega_deep_learning.csv")
print("Real-world shape:", df_rw.shape)
print(df_rw["isMalicious"].value_counts())


## 3. Harmonize labels

PhiUSIIL uses `label=1` for **legitimate** and `label=0` for **phishing** — the opposite of the real-world dataset (`isMalicious=1` for malicious). Flip PhiUSIIL so `1 = malicious` in both. This is the convention the downstream app (`app.py`) expects.

Without this step, concatenating the two datasets produces contradictory training signal and the model ends up inverting the meaning of its output classes.


In [ ]:
df_phi["label"] = 1 - df_phi["label"]
print("PhiUSIIL after flip (now 1 = malicious):")
print(df_phi["label"].value_counts())


## 4. Engineered URL features (11)

`fe_is_https` has been removed as a robustness ablation — every PhiUSIIL URL starts with `https://` and no real-world URL does, so the feature acted as a dataset fingerprint rather than a phishing signal.

This extractor **must** match `feature_extractor.py` in the app folder. If you change it here, update `feature_extractor.py` and `FEATURE_COLUMNS` to match, and re-save the model.


In [ ]:
SUSPICIOUS_TOKENS = ["login", "verify", "secure", "account", "update", "confirm", "bank"]

FEATURE_COLUMNS = [
    "fe_url_length",
    "fe_num_special_chars",
    "fe_has_suspicious_token",
    "fe_digit_ratio",
    "fe_entropy",
    "fe_num_subdomains",
    "fe_is_ip",
    "fe_domain_length",
    "fe_hyphen_count",
    "fe_at_symbol",
    "fe_path_depth",
]


def extract_url_features(url):
    url = str(url)
    parsed = urlparse(url)
    hostname = parsed.hostname or ""
    path = parsed.path or ""

    length = len(url)
    num_special = sum(url.count(c) for c in ["@", "!", "$", "%", "#"])
    has_suspicious = int(any(t in url.lower() for t in SUSPICIOUS_TOKENS))
    digit_ratio = sum(c.isdigit() for c in url) / (length + 1)

    if length > 0:
        char_freq = {c: url.count(c) / length for c in set(url)}
        entropy = -sum(f * np.log2(f) for f in char_freq.values() if f > 0)
    else:
        entropy = 0.0

    return {
        "fe_url_length": length,
        "fe_num_special_chars": num_special,
        "fe_has_suspicious_token": has_suspicious,
        "fe_digit_ratio": digit_ratio,
        "fe_entropy": entropy,
        "fe_num_subdomains": hostname.count("."),
        "fe_is_ip": int(bool(re.match(r"^\d{1,3}(\.\d{1,3}){3}$", hostname))),
        "fe_domain_length": len(hostname),
        "fe_hyphen_count": hostname.count("-"),
        "fe_at_symbol": int("@" in url),
        "fe_path_depth": path.count("/"),
    }


## 5. Extract features from both datasets

In [ ]:
print("Extracting features from PhiUSIIL URLs...")
X_phi = df_phi["URL"].apply(extract_url_features).apply(pd.Series)[FEATURE_COLUMNS]
y_phi = df_phi["label"].astype(int)

print("Extracting features from real-world URLs...")
X_rw = df_rw["url"].apply(extract_url_features).apply(pd.Series)[FEATURE_COLUMNS]
y_rw = df_rw["isMalicious"].astype(int)

print(f"PhiUSIIL: {X_phi.shape}, Real-world: {X_rw.shape}")


## 6. In-domain baseline: LR on PhiUSIIL only

Sanity check that the engineered features are informative within a single dataset.


In [ ]:
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X_phi, y_phi, test_size=0.2, random_state=42, stratify=y_phi
)

lr_phi = LogisticRegression(max_iter=1000)
lr_phi.fit(Xp_tr, yp_tr)
acc_phi_only = accuracy_score(yp_te, lr_phi.predict(Xp_te))
print(f"PhiUSIIL-only LR accuracy: {acc_phi_only:.4f}")


## 7. Cross-dataset generalization

Train on PhiUSIIL, evaluate on the real-world dataset. This measures how well features learned from one URL distribution transfer to another — a more honest test than in-domain accuracy.


In [ ]:
acc_cross = accuracy_score(y_rw, lr_phi.predict(X_rw))
print(f"PhiUSIIL-trained LR on real-world data: {acc_cross:.4f}\n")
print(classification_report(y_rw, lr_phi.predict(X_rw), digits=3))


## 8. Retrain LR on both datasets (for comparison)

In [ ]:
X_all = pd.concat([X_phi, X_rw], ignore_index=True)
y_all = pd.concat([y_phi, y_rw], ignore_index=True)

Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

lr_both = LogisticRegression(max_iter=1000)
lr_both.fit(Xa_tr, ya_tr)
acc_lr_both = accuracy_score(ya_te, lr_both.predict(Xa_te))
print(f"Combined LR accuracy: {acc_lr_both:.4f}")


## 9. Train Random Forest (final model)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(Xa_tr, ya_tr)

rf_pred = rf.predict(Xa_te)
acc_rf_both = accuracy_score(ya_te, rf_pred)

print(f"Combined RF accuracy: {acc_rf_both:.4f}\n")
print(classification_report(ya_te, rf_pred, digits=3))

print("\nFeature importances (sorted):")
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=False)
print(importances)


## 10. Save the model

Saved to `models/rf_model.pkl` — matches the `MODEL_PATH` in `app.py`.


In [ ]:
import joblib

os.makedirs("models", exist_ok=True)
joblib.dump(rf, "models/rf_model.pkl")
print("Saved models/rf_model.pkl")


## 11. Presentation charts

The numbers below pull live from the variables computed above, so they stay in sync if you retrain.


In [ ]:
# Chart 1: Accuracy across stages
stages = [
    "PhiUSIIL only\n(LR, in-domain)",
    "PhiUSIIL → Real-world\n(LR, cross-domain)",
    "Both datasets\n(LR)",
    "Both datasets\n(Random Forest)",
]
accuracies = [acc_phi_only * 100, acc_cross * 100, acc_lr_both * 100, acc_rf_both * 100]
colors = ["#43A047", "#E53935", "#FB8C00", "#1E88E5"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stages, accuracies, color=colors, width=0.5, edgecolor="white")
ax.set_ylim(0, 110)
ax.set_ylabel("Accuracy (%)", fontsize=12)
ax.set_title("Model Accuracy Across Evaluation Stages", fontsize=14, fontweight="bold")
ax.axhline(y=80, color="gray", linestyle="--", linewidth=1, label="80% goal")

for bar, val in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f"{val:.1f}%", ha="center", fontsize=11, fontweight="bold")

ax.legend()
plt.tight_layout()
plt.savefig("accuracy_stages.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Chart 2: Feature importance
top10 = importances.head(10)
fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ["#1E88E5" if v >= 0.05 else "#90CAF9" for v in top10.values]
ax.barh(top10.index[::-1], top10.values[::-1], color=bar_colors[::-1], edgecolor="white")
ax.set_xlabel("Feature importance", fontsize=12)
ax.set_title("Random Forest Feature Importance", fontsize=14, fontweight="bold")
ax.axvline(x=0.05, color="gray", linestyle="--", linewidth=1, label="5% threshold")
ax.legend()
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Chart 3: Confusion matrix
cm = confusion_matrix(ya_te, rf_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=["Benign", "Malicious"]).plot(
    ax=ax, colorbar=False, cmap="Blues"
)
ax.set_title(
    f"Random Forest — Both Datasets\n(Accuracy: {acc_rf_both*100:.1f}%)",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
plt.savefig("confusion_matrix_rf.png", dpi=150, bbox_inches="tight")
plt.show()
